<a id="top"></a>

# Lab L1406: graph state and reducers

```yaml
title: "Lab L1406: graph state and reducers"
keywords: langgraph, state, reducer, add_messages, message history, invariant, overwrite, recursion limit, state boundary, dependency, lab
estimated duration: 30
```

> **Lesson:** L14. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L14/objectives.md).
> **Solutions:** `L1206_lab_solutions.ipynb`.
> Runs **fully offline** -- a deterministic `StubChat` + the REAL `ToolNode`. No API key needed.

**You will:** reason about graph **state** -- define the `messages` reducer that preserves the L10
`tool_use`/`tool_result` invariant, *break and fix* it, and draw the state-vs-dependency boundary
(objective 3).

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- Define the state](#2-problem-1----define-the-state)
- [3. Problem 2 -- Watch the history grow](#3-problem-2----watch-the-history-grow)
- [4. Problem 3 -- Break it, then fix the reducer](#4-problem-3----break-it-then-fix-the-reducer)
- [5. Problem 4 -- State or dependency? (written)](#5-problem-4----state-or-dependency-written)
- [6. Self-check](#6-self-check)

## 1. Setup

Given: the shared tools, the offline `StubChat`, and a `build_agent(state_cls)` helper that wires
the *same* shallow agent over whatever **state schema** you pass. Run this cell first.

In [ ]:
from typing import Annotated, Any, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.tools import StructuredTool
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common import tools as T

calculator = StructuredTool.from_function(T.calculator, name="calculator", description="arithmetic")
lookup = StructuredTool.from_function(T.lookup, name="lookup", description="city population")
LC_TOOLS = [calculator, lookup]


class StubChat:
    """Offline, scripted stand-in: turn 0 -> calculator, turn 1 -> lookup, then a final answer."""

    def bind_tools(self, tools: list[object]) -> "StubChat":
        return self

    def invoke(self, messages: list[BaseMessage]) -> AIMessage:
        ai_turns = sum(1 for m in messages if isinstance(m, AIMessage))
        if ai_turns == 0:
            return AIMessage(
                content="",
                tool_calls=[{"name": "calculator", "args": {"expression": "21*2"}, "id": "c1"}],
            )
        if ai_turns == 1:
            return AIMessage(
                content="", tool_calls=[{"name": "lookup", "args": {"city": "Tokyo"}, "id": "l1"}]
            )
        return AIMessage(content="21*2 = 42, and Tokyo's population is 37000000.")


model = StubChat()
TASK = "Use the calculator to compute 21 * 2, then tell me the population of Tokyo."


def build_agent(state_cls: type) -> Any:
    """Wire the SAME shallow agent over whatever state schema you pass. Only the state's
    `messages` reducer will differ between the problems below -- nothing else."""

    def agent_node(state: Any) -> dict[str, object]:
        return {
            "messages": [model.bind_tools(LC_TOOLS).invoke(state["messages"])],
            "step": state["step"] + 1,
        }

    def route(state: Any) -> str:
        last = state["messages"][-1]
        return "tools" if isinstance(last, AIMessage) and last.tool_calls else END

    b = StateGraph(state_cls)
    b.add_node("agent", agent_node)
    b.add_node("tools", ToolNode(LC_TOOLS, handle_tool_errors=True))
    b.set_entry_point("agent")
    b.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    b.add_edge("tools", "agent")
    return b.compile()

[↑ Back to top](#top)

## 2. Problem 1 -- Define the state

Define `AgentState` so the agent's history **accumulates**: `messages` with the `add_messages`
**append** reducer, plus a `step: int` counter (default overwrite reducer).

In [ ]:
# TODO: complete AgentState so the agent's history ACCUMULATES.
#   - messages: Annotated[list[BaseMessage], add_messages]   <-- the APPEND reducer
#   - step: int
# The stub below is incomplete -- REPLACE it.
class AgentState(TypedDict):
    messages: list[BaseMessage]

[↑ Back to top](#top)

## 3. Problem 2 -- Watch the history grow

Build the agent over `AgentState`, invoke it on `TASK`, and print each message (type + the tools it
asked for) and the final step count. You should see the history grow: `Human` -> `AI(calculator)`
-> `Tool` -> `AI(lookup)` -> `Tool` -> `AI(text)`.

In [ ]:
# TODO: build the agent over AgentState, invoke it on TASK (step=0 in the initial state),
#        and print each message's type + the tools it asked for, then the final step count.
#   app = build_agent(AgentState)
#   final = app.invoke({"messages": [HumanMessage(TASK)], "step": 0})
#   for m in final["messages"]: print(type(m).__name__, ...)
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 -- Break it, then fix the reducer

First, **run the broken version below** (given): `BrokenState` uses a plain `list[BaseMessage]`,
so the default reducer **overwrites** the history each turn. The `tool_use`/`tool_result` pairing
breaks, the loop never reaches `END`, and it hits the recursion cap -- the L10 invariant bug, reborn.

Then **fix it**: write `FixedState` identical to `BrokenState` but with the `add_messages` reducer,
and confirm the agent runs to completion.

In [ ]:
class BrokenState(TypedDict):
    messages: list[BaseMessage]  # NO add_messages -> default OVERWRITE reducer (the bug)
    step: int


broken_app = build_agent(BrokenState)
try:
    broken_app.invoke({"messages": [HumanMessage(TASK)], "step": 0}, {"recursion_limit": 8})
except GraphRecursionError as exc:
    print("broken ->", str(exc).split(".")[0], "(history clobbered; pairing broke)")

In [ ]:
# TODO: write FixedState -- the SAME as BrokenState but with the add_messages reducer on
#        `messages`. Then build_agent(FixedState), invoke on TASK, and print the history length
#        and the final text. It should run to completion (no recursion error).
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 5. Problem 4 -- State or dependency? (written)

_Double-click to edit._ For each item, say **state** (data that flows between nodes) or
**dependency** (wired in at build time), and one line why.

1. `messages` -- ?
2. `step` -- ?
3. the `ChatAnthropic` / `StubChat` client -- ?
4. `LC_TOOLS` (the tool registry) -- ?
5. the `ANTHROPIC_API_KEY` -- ?

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L1206_lab_solutions.ipynb`. You're done when: `AgentState` uses `add_messages` on
`messages`; Problem 2 shows a 6-message history and `steps: 3`; `BrokenState` raises
`GraphRecursionError` while your `FixedState` runs to completion; and you can classify each item in
Problem 4 as **state** (flows between nodes) or **dependency** (wired in at build time).

[↑ Back to top](#top)